## 01. IMPORTS

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


print("Model-ready data preparation libraries imported successfully.")

Model-ready data preparation libraries imported successfully.


## 02. FILE LOCATIONS AND DATA LOADING

In [2]:
PROJECT_ROOT = Path(
    r"C:\Users\bongo\OneDrive\Desktop\GitHub\Movie Analysis Predictor"
)

FEATURES_DIR = (
    PROJECT_ROOT
    / "Data"
    / "Features"
)

MODEL_READY_DIR = (
    PROJECT_ROOT
    / "Data"
    / "Model_Ready"
)

MODEL_READY_DIR.mkdir(
    parents=True,
    exist_ok=True
)

HISTORICAL_FEATURES_PATH = (
    FEATURES_DIR
    / "historical_model_features.csv"
)

PREDICTION_FEATURES_PATH = (
    FEATURES_DIR
    / "prediction_target_features.csv"
)


historical = pd.read_csv(
    HISTORICAL_FEATURES_PATH,
    low_memory=False
)

prediction = pd.read_csv(
    PREDICTION_FEATURES_PATH,
    low_memory=False
)


print("=" * 80)
print("MODEL-READY DATA INPUTS")
print("=" * 80)

print(
    f"\nHistorical dataset: "
    f"{historical.shape[0]:,} rows × "
    f"{historical.shape[1]:,} columns"
)

print(
    f"Prediction dataset: "
    f"{prediction.shape[0]:,} rows × "
    f"{prediction.shape[1]:,} columns"
)

print("\nHistorical file:")
print(HISTORICAL_FEATURES_PATH)

print("\nPrediction file:")
print(PREDICTION_FEATURES_PATH)

MODEL-READY DATA INPUTS

Historical dataset: 7,479 rows × 17 columns
Prediction dataset: 19 rows × 18 columns

Historical file:
C:\Users\bongo\OneDrive\Desktop\GitHub\Movie Analysis Predictor\Data\Features\historical_model_features.csv

Prediction file:
C:\Users\bongo\OneDrive\Desktop\GitHub\Movie Analysis Predictor\Data\Features\prediction_target_features.csv


## 03. POST-EDA DATA VALIDATION

In [3]:
print("=" * 80)
print("POST-EDA VALIDATION")
print("=" * 80)


checks = {
    "Historical rows": len(historical),
    "Historical columns": historical.shape[1],

    "Missing regression targets":
        historical[
            "worldwide_box_office"
        ].isna().sum(),

    "Missing log regression targets":
        historical[
            "log_worldwide_box_office"
        ].isna().sum(),

    "Unique genres":
        historical[
            "primary_genre"
        ].nunique(dropna=True),

    "Unknown genres":
        historical[
            "primary_genre"
        ].eq("Unknown").sum(),

    "Duplicate historical rows":
        historical.duplicated().sum(),

    "Duplicate historical titles/year":
        historical.duplicated(
            subset=[
                "title",
                "release_year"
            ]
        ).sum()
}

for name, value in checks.items():
    print(f"{name:<35}: {value:,}")


# ------------------------------------------------------------
# Critical assertions
# ------------------------------------------------------------

assert historical[
    "worldwide_box_office"
].isna().sum() == 0

assert historical[
    "log_worldwide_box_office"
].isna().sum() == 0

assert historical[
    "primary_genre"
].nunique() > 1

assert historical[
    "primary_genre"
].eq("Unknown").sum() == 0


print(
    "\nCritical validation checks passed."
)

POST-EDA VALIDATION
Historical rows                    : 7,479
Historical columns                 : 17
Missing regression targets         : 0
Missing log regression targets     : 0
Unique genres                      : 18
Unknown genres                     : 0
Duplicate historical rows          : 0
Duplicate historical titles/year   : 0

Critical validation checks passed.


## 04. CANDIDATE MODEL FEATURE INVENTORY

In [5]:
regression_target = (
    "log_worldwide_box_office"
)

identifier_columns = [
    "title",
    "release_date"
]

numeric_candidates = [
    "log_budget",

    "release_year",
    "release_month",
    "release_quarter",
    "holiday_release",

    "director_prior_films",
    "director_prior_avg_gross",

    "star_prior_films",
    "star_prior_avg_gross",

    "company_prior_films",
    "company_prior_avg_gross"
]

categorical_candidates = [
    "primary_genre",
    "release_season"
]


all_candidates = (
    numeric_candidates
    + categorical_candidates
)

availability = pd.DataFrame({
    "feature": all_candidates,
    "available_historical": [
        feature in historical.columns
        for feature in all_candidates
    ],
    "available_prediction": [
        feature in prediction.columns
        for feature in all_candidates
    ]
})

print("=" * 80)
print("CANDIDATE FEATURE INVENTORY")
print("=" * 80)

display(availability)

CANDIDATE FEATURE INVENTORY


,feature,available_historical,available_prediction
0,log_budget,True,True
1,release_year,True,True
2,release_month,True,True
3,release_quarter,True,True
4,holiday_release,True,True
5,director_prior_films,True,True
6,director_prior_avg_gross,True,True
7,star_prior_films,True,True
8,star_prior_avg_gross,True,True
9,company_prior_films,True,True


## 05. BLOCKBUSTER TARGET AUDIT

In [6]:
box_office = historical[
    "worldwide_box_office"
]

threshold_options = {
    "$100M+": 100_000_000,
    "$250M+": 250_000_000,
    "$500M+": 500_000_000,
    "$1B+": 1_000_000_000,
    "Top 20%": box_office.quantile(0.80),
    "Top 10%": box_office.quantile(0.90),
    "Top 5%": box_office.quantile(0.95)
}

blockbuster_audit = []

for definition, threshold in threshold_options.items():

    blockbuster_count = (
        box_office >= threshold
    ).sum()

    non_blockbuster_count = (
        box_office < threshold
    ).sum()

    blockbuster_percent = (
        blockbuster_count
        / len(box_office)
        * 100
    )

    blockbuster_audit.append({
        "definition": definition,
        "threshold_usd": threshold,
        "blockbusters": blockbuster_count,
        "non_blockbusters": non_blockbuster_count,
        "blockbuster_percent": blockbuster_percent
    })


blockbuster_audit = pd.DataFrame(
    blockbuster_audit
)

blockbuster_audit[
    "threshold_usd"
] = (
    blockbuster_audit[
        "threshold_usd"
    ]
    .round(0)
)

blockbuster_audit[
    "blockbuster_percent"
] = (
    blockbuster_audit[
        "blockbuster_percent"
    ]
    .round(2)
)

print("=" * 80)
print("BLOCKBUSTER TARGET DEFINITION AUDIT")
print("=" * 80)

display(blockbuster_audit)

BLOCKBUSTER TARGET DEFINITION AUDIT


,definition,threshold_usd,blockbusters,non_blockbusters,blockbuster_percent
0,$100M+,1.000000e+08,1525,5954,20.39
1,$250M+,2.500000e+08,594,6885,7.94
2,$500M+,5.000000e+08,207,7272,2.77
3,$1B+,1.000000e+09,48,7431,0.64
4,Top 20%,1.018482e+08,1496,5983,20.00
5,Top 10%,2.090678e+08,748,6731,10.00
6,Top 5%,3.528074e+08,374,7105,5.00


## 06. CREATE BLOCKBUSTER CLASSIFICATION TARGET

In [7]:
BLOCKBUSTER_QUANTILE = 0.90

BLOCKBUSTER_THRESHOLD = (
    historical["worldwide_box_office"]
    .quantile(BLOCKBUSTER_QUANTILE)
)

historical["blockbuster"] = (
    historical["worldwide_box_office"]
    >= BLOCKBUSTER_THRESHOLD
).astype(int)


blockbuster_counts = (
    historical["blockbuster"]
    .value_counts()
    .sort_index()
)

blockbuster_percent = (
    historical["blockbuster"]
    .value_counts(normalize=True)
    .sort_index()
    * 100
)


print("=" * 80)
print("BLOCKBUSTER CLASSIFICATION TARGET")
print("=" * 80)

print(
    f"\nDefinition: Top "
    f"{int((1 - BLOCKBUSTER_QUANTILE) * 100)}% "
    f"of historical worldwide box office"
)

print(
    f"Threshold: "
    f"${BLOCKBUSTER_THRESHOLD:,.0f}"
)

print("\nClass distribution:")

print(
    f"Non-blockbuster (0): "
    f"{blockbuster_counts.get(0, 0):,} "
    f"({blockbuster_percent.get(0, 0):.2f}%)"
)

print(
    f"Blockbuster (1):     "
    f"{blockbuster_counts.get(1, 0):,} "
    f"({blockbuster_percent.get(1, 0):.2f}%)"
)

assert historical["blockbuster"].isna().sum() == 0
assert historical["blockbuster"].nunique() == 2

print("\nClassification target created successfully.")

BLOCKBUSTER CLASSIFICATION TARGET

Definition: Top 9% of historical worldwide box office
Threshold: $209,067,794

Class distribution:
Non-blockbuster (0): 6,731 (90.00%)
Blockbuster (1):     748 (10.00%)

Classification target created successfully.


## 07. DEFINE FINAL MODEL FEATURE SET

In [8]:
columns_to_exclude = [
    "title",
    "release_date",

    # Redundant with release_month
    "release_quarter",

    # Outcome columns — never predictors
    "worldwide_box_office",
    "log_worldwide_box_office",
    "blockbuster"
]


final_numeric_features = [
    "log_budget",

    "release_year",
    "release_month",
    "holiday_release",

    "director_prior_films",
    "director_prior_avg_gross",

    "star_prior_films",
    "star_prior_avg_gross",

    "company_prior_films",
    "company_prior_avg_gross"
]


final_categorical_features = [
    "primary_genre",
    "release_season"
]


final_model_features = (
    final_numeric_features
    + final_categorical_features
)


print("=" * 80)
print("FINAL MODEL FEATURE SET")
print("=" * 80)

print(
    f"\nNumerical features: "
    f"{len(final_numeric_features)}"
)

for feature in final_numeric_features:
    print(f"  - {feature}")


print(
    f"\nCategorical features: "
    f"{len(final_categorical_features)}"
)

for feature in final_categorical_features:
    print(f"  - {feature}")


print(
    f"\nTotal predictor features: "
    f"{len(final_model_features)}"
)


# Validate availability in both datasets
missing_historical = [
    feature
    for feature in final_model_features
    if feature not in historical.columns
]

missing_prediction = [
    feature
    for feature in final_model_features
    if feature not in prediction.columns
]

assert len(missing_historical) == 0, (
    f"Missing historical features: "
    f"{missing_historical}"
)

assert len(missing_prediction) == 0, (
    f"Missing prediction features: "
    f"{missing_prediction}"
)

print(
    "\nAll selected features are available "
    "in both datasets."
)

FINAL MODEL FEATURE SET

Numerical features: 10
  - log_budget
  - release_year
  - release_month
  - holiday_release
  - director_prior_films
  - director_prior_avg_gross
  - star_prior_films
  - star_prior_avg_gross
  - company_prior_films
  - company_prior_avg_gross

Categorical features: 2
  - primary_genre
  - release_season

Total predictor features: 12

All selected features are available in both datasets.


## 08. CREATE TRACK-RECORD HISTORY INDICATORS

In [9]:
datasets = {
    "Historical": historical,
    "Prediction": prediction
}


for dataset_name, df in datasets.items():

    df["director_has_prior_history"] = (
        df["director_prior_films"] > 0
    ).astype(int)

    df["star_has_prior_history"] = (
        df["star_prior_films"] > 0
    ).astype(int)

    df["company_has_prior_history"] = (
        df["company_prior_films"] > 0
    ).astype(int)


print("=" * 80)
print("TRACK-RECORD HISTORY INDICATORS")
print("=" * 80)

indicator_columns = [
    "director_has_prior_history",
    "star_has_prior_history",
    "company_has_prior_history"
]

for column in indicator_columns:

    print(f"\n{column}")

    print(
        historical[column]
        .value_counts()
        .sort_index()
    )


# Add indicators to final numerical features
final_numeric_features.extend(
    indicator_columns
)

final_model_features = (
    final_numeric_features
    + final_categorical_features
)

TRACK-RECORD HISTORY INDICATORS

director_has_prior_history
director_has_prior_history
0    2872
1    4607
Name: count, dtype: int64

star_has_prior_history
star_has_prior_history
0    2802
1    4677
Name: count, dtype: int64

company_has_prior_history
company_has_prior_history
0    2480
1    4999
Name: count, dtype: int64


## 09. CREATE BUDGET MISSINGNESS INDICATOR

In [10]:
historical["budget_missing"] = (
    historical["log_budget"]
    .isna()
    .astype(int)
)

prediction["budget_missing"] = (
    prediction["log_budget"]
    .isna()
    .astype(int)
)


if "budget_missing" not in final_numeric_features:
    final_numeric_features.append(
        "budget_missing"
    )

final_model_features = (
    final_numeric_features
    + final_categorical_features
)


print("=" * 80)
print("BUDGET MISSINGNESS INDICATOR")
print("=" * 80)

print("\nHistorical:")
print(
    historical["budget_missing"]
    .value_counts()
    .sort_index()
)

print("\nPrediction:")
print(
    prediction["budget_missing"]
    .value_counts()
    .sort_index()
)

print(
    f"\nFinal predictor count: "
    f"{len(final_model_features)}"
)

BUDGET MISSINGNESS INDICATOR

Historical:
budget_missing
0    5436
1    2043
Name: count, dtype: int64

Prediction:
budget_missing
0    10
1     9
Name: count, dtype: int64

Final predictor count: 16


## 10. CHRONOLOGICAL TRAIN/VALIDATION/TEST SPLIT

In [11]:
TRAIN_END_YEAR = 2015

VALIDATION_START_YEAR = 2016
VALIDATION_END_YEAR = 2018

TEST_START_YEAR = 2019
TEST_END_YEAR = 2020


# ------------------------------------------------------------
# Create chronological subsets
# ------------------------------------------------------------

train_df = historical[
    historical["release_year"] <= TRAIN_END_YEAR
].copy()

validation_df = historical[
    historical["release_year"].between(
        VALIDATION_START_YEAR,
        VALIDATION_END_YEAR
    )
].copy()

test_df = historical[
    historical["release_year"].between(
        TEST_START_YEAR,
        TEST_END_YEAR
    )
].copy()


print("=" * 80)
print("CHRONOLOGICAL DATA SPLIT")
print("=" * 80)

print(
    f"\nTraining period: "
    f"{train_df['release_year'].min()}–"
    f"{train_df['release_year'].max()}"
)

print(
    f"Training rows: "
    f"{len(train_df):,}"
)

print(
    f"\nValidation period: "
    f"{validation_df['release_year'].min()}–"
    f"{validation_df['release_year'].max()}"
)

print(
    f"Validation rows: "
    f"{len(validation_df):,}"
)

print(
    f"\nTest period: "
    f"{test_df['release_year'].min()}–"
    f"{test_df['release_year'].max()}"
)

print(
    f"Test rows: "
    f"{len(test_df):,}"
)

print(
    f"\nTotal split rows: "
    f"{len(train_df) + len(validation_df) + len(test_df):,}"
)

assert (
    len(train_df)
    + len(validation_df)
    + len(test_df)
    == len(historical)
)

print(
    "\nChronological split completed successfully."
)

CHRONOLOGICAL DATA SPLIT

Training period: 1980–2015
Training rows: 6,668

Validation period: 2016–2018
Validation rows: 600

Test period: 2019–2020
Test rows: 211

Total split rows: 7,479

Chronological split completed successfully.


## 11. FINALISE BLOCKBUSTER TARGET

In [13]:
BLOCKBUSTER_QUANTILE = 0.90

BLOCKBUSTER_THRESHOLD = (
    train_df["worldwide_box_office"]
    .quantile(BLOCKBUSTER_QUANTILE)
)


for df in [
    train_df,
    validation_df,
    test_df
]:
    df["blockbuster"] = (
        df["worldwide_box_office"]
        >= BLOCKBUSTER_THRESHOLD
    ).astype(int)


print("=" * 80)
print("FINAL BLOCKBUSTER TARGET")
print("=" * 80)

print(
    f"\nTraining-derived threshold: "
    f"${BLOCKBUSTER_THRESHOLD:,.0f}"
)


for name, df in {
    "Training": train_df,
    "Validation": validation_df,
    "Test": test_df
}.items():

    positive = (
        df["blockbuster"] == 1
    ).sum()

    percent = (
        positive
        / len(df)
        * 100
    )

    print(
        f"\n{name}: "
        f"{positive:,} blockbusters "
        f"({percent:.2f}%)"
    )

FINAL BLOCKBUSTER TARGET

Training-derived threshold: $183,907,650

Training: 667 blockbusters (10.00%)

Validation: 129 blockbusters (21.50%)

Test: 50 blockbusters (23.70%)


## 12. CREATE FEATURES AND TARGETS

In [14]:
# ------------------------------------------------------------
# Regression target
# ------------------------------------------------------------

REGRESSION_TARGET = (
    "log_worldwide_box_office"
)

# ------------------------------------------------------------
# Classification target
# ------------------------------------------------------------

CLASSIFICATION_TARGET = (
    "blockbuster"
)


# ------------------------------------------------------------
# Feature matrices
# ------------------------------------------------------------

X_train = train_df[
    final_model_features
].copy()

X_validation = validation_df[
    final_model_features
].copy()

X_test = test_df[
    final_model_features
].copy()

X_prediction = prediction[
    final_model_features
].copy()


# ------------------------------------------------------------
# Regression targets
# ------------------------------------------------------------

y_train_reg = train_df[
    REGRESSION_TARGET
].copy()

y_validation_reg = validation_df[
    REGRESSION_TARGET
].copy()

y_test_reg = test_df[
    REGRESSION_TARGET
].copy()


# ------------------------------------------------------------
# Classification targets
# ------------------------------------------------------------

y_train_cls = train_df[
    CLASSIFICATION_TARGET
].copy()

y_validation_cls = validation_df[
    CLASSIFICATION_TARGET
].copy()

y_test_cls = test_df[
    CLASSIFICATION_TARGET
].copy()


print("=" * 80)
print("FEATURE AND TARGET MATRICES")
print("=" * 80)

print(
    f"\nX_train:      "
    f"{X_train.shape}"
)

print(
    f"X_validation: "
    f"{X_validation.shape}"
)

print(
    f"X_test:       "
    f"{X_test.shape}"
)

print(
    f"X_prediction: "
    f"{X_prediction.shape}"
)

print(
    f"\nRegression training targets: "
    f"{len(y_train_reg):,}"
)

print(
    f"Classification training targets: "
    f"{len(y_train_cls):,}"
)

FEATURE AND TARGET MATRICES

X_train:      (6668, 16)
X_validation: (600, 16)
X_test:       (211, 16)
X_prediction: (19, 16)

Regression training targets: 6,668
Classification training targets: 6,668


## DEFINE PREPROCESSING PIPELINE

In [15]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)


# ------------------------------------------------------------
# Numerical preprocessing
# ------------------------------------------------------------

numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)


# ------------------------------------------------------------
# Categorical preprocessing
# ------------------------------------------------------------

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)


# ------------------------------------------------------------
# Combined preprocessor
# ------------------------------------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            final_numeric_features
        ),
        (
            "categorical",
            categorical_pipeline,
            final_categorical_features
        )
    ],
    remainder="drop"
)


print("=" * 80)
print("PREPROCESSING PIPELINE")
print("=" * 80)

print(
    f"\nNumerical features: "
    f"{len(final_numeric_features)}"
)

print(
    f"Categorical features: "
    f"{len(final_categorical_features)}"
)

print(
    "\nPreprocessing pipeline created successfully."
)

PREPROCESSING PIPELINE

Numerical features: 14
Categorical features: 2

Preprocessing pipeline created successfully.


## 14. FIT PREPROCESSOR ON TRAINING DATA

In [16]:
X_train_processed = (
    preprocessor.fit_transform(
        X_train
    )
)

X_validation_processed = (
    preprocessor.transform(
        X_validation
    )
)

X_test_processed = (
    preprocessor.transform(
        X_test
    )
)

X_prediction_processed = (
    preprocessor.transform(
        X_prediction
    )
)


print("=" * 80)
print("PROCESSED FEATURE MATRICES")
print("=" * 80)

print(
    f"\nTraining:   "
    f"{X_train_processed.shape}"
)

print(
    f"Validation: "
    f"{X_validation_processed.shape}"
)

print(
    f"Test:       "
    f"{X_test_processed.shape}"
)

print(
    f"Prediction: "
    f"{X_prediction_processed.shape}"
)

PROCESSED FEATURE MATRICES

Training:   (6668, 36)
Validation: (600, 36)
Test:       (211, 36)
Prediction: (19, 36)


## 15. RECOVER PROCESSED FEATURE NAMES

In [17]:
processed_feature_names = (
    preprocessor
    .get_feature_names_out()
)

processed_feature_names = [
    feature
    .replace("numeric__", "")
    .replace("categorical__", "")
    for feature in processed_feature_names
]


print("=" * 80)
print("PROCESSED FEATURE INVENTORY")
print("=" * 80)

print(
    f"\nFinal processed feature count: "
    f"{len(processed_feature_names)}"
)

for i, feature in enumerate(
    processed_feature_names,
    start=1
):
    print(
        f"{i:>3}. {feature}"
    )

PROCESSED FEATURE INVENTORY

Final processed feature count: 36
  1. log_budget
  2. release_year
  3. release_month
  4. holiday_release
  5. director_prior_films
  6. director_prior_avg_gross
  7. star_prior_films
  8. star_prior_avg_gross
  9. company_prior_films
 10. company_prior_avg_gross
 11. director_has_prior_history
 12. star_has_prior_history
 13. company_has_prior_history
 14. budget_missing
 15. primary_genre_Action
 16. primary_genre_Adventure
 17. primary_genre_Animation
 18. primary_genre_Biography
 19. primary_genre_Comedy
 20. primary_genre_Crime
 21. primary_genre_Drama
 22. primary_genre_Family
 23. primary_genre_Fantasy
 24. primary_genre_Horror
 25. primary_genre_Music
 26. primary_genre_Mystery
 27. primary_genre_Romance
 28. primary_genre_Sci-Fi
 29. primary_genre_Thriller
 30. primary_genre_Western
 31. release_season_Unknown
 32. release_season_fall
 33. release_season_holiday
 34. release_season_spring
 35. release_season_summer
 36. release_season_winter


## 16. CONVERT PROCESSED ARRAYS TO DATAFRAMES

In [18]:
X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=processed_feature_names,
    index=X_train.index
)

X_validation_processed = pd.DataFrame(
    X_validation_processed,
    columns=processed_feature_names,
    index=X_validation.index
)

X_test_processed = pd.DataFrame(
    X_test_processed,
    columns=processed_feature_names,
    index=X_test.index
)

X_prediction_processed = pd.DataFrame(
    X_prediction_processed,
    columns=processed_feature_names,
    index=X_prediction.index
)


print("=" * 80)
print("PROCESSED DATAFRAME SHAPES")
print("=" * 80)

print(
    "Train:",
    X_train_processed.shape
)

print(
    "Validation:",
    X_validation_processed.shape
)

print(
    "Test:",
    X_test_processed.shape
)

print(
    "Prediction:",
    X_prediction_processed.shape
)

PROCESSED DATAFRAME SHAPES
Train: (6668, 36)
Validation: (600, 36)
Test: (211, 36)
Prediction: (19, 36)


## 17. FINAL PROCESSED-DATA QUALITY VALIDATION

In [19]:
processed_datasets = {
    "Training": X_train_processed,
    "Validation": X_validation_processed,
    "Test": X_test_processed,
    "Prediction": X_prediction_processed
}

quality_results = []

for name, df in processed_datasets.items():

    missing_values = int(
        df.isna().sum().sum()
    )

    infinite_values = int(
        np.isinf(
            df.to_numpy(dtype=float)
        ).sum()
    )

    quality_results.append({
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "missing_values": missing_values,
        "infinite_values": infinite_values
    })


quality_results = pd.DataFrame(
    quality_results
)

print("=" * 80)
print("FINAL MODEL-READY DATA QUALITY CHECK")
print("=" * 80)

display(quality_results)


# ------------------------------------------------------------
# Target validation
# ------------------------------------------------------------

assert len(X_train_processed) == len(y_train_reg)
assert len(X_validation_processed) == len(y_validation_reg)
assert len(X_test_processed) == len(y_test_reg)

assert len(X_train_processed) == len(y_train_cls)
assert len(X_validation_processed) == len(y_validation_cls)
assert len(X_test_processed) == len(y_test_cls)

assert y_train_reg.isna().sum() == 0
assert y_validation_reg.isna().sum() == 0
assert y_test_reg.isna().sum() == 0

assert y_train_cls.isna().sum() == 0
assert y_validation_cls.isna().sum() == 0
assert y_test_cls.isna().sum() == 0


for name, df in processed_datasets.items():

    assert df.isna().sum().sum() == 0, (
        f"{name} contains missing values."
    )

    assert np.isfinite(
        df.to_numpy(dtype=float)
    ).all(), (
        f"{name} contains infinite values."
    )


print(
    "\nAll processed datasets passed "
    "model-readiness validation."
)

FINAL MODEL-READY DATA QUALITY CHECK


,dataset,rows,columns,missing_values,infinite_values
0,Training,6668,36,0,0
1,Validation,600,36,0,0
2,Test,211,36,0,0
3,Prediction,19,36,0,0



All processed datasets passed model-readiness validation.


## 18. SAVE MODEL-READY FEATURE DATASETS

In [20]:
FEATURE_OUTPUT_DIR = (
    MODEL_READY_DIR
    / "Features"
)

TARGET_OUTPUT_DIR = (
    MODEL_READY_DIR
    / "Targets"
)

METADATA_OUTPUT_DIR = (
    MODEL_READY_DIR
    / "Metadata"
)

FEATURE_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TARGET_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

METADATA_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# Save processed feature matrices
# ------------------------------------------------------------

X_train_processed.to_csv(
    FEATURE_OUTPUT_DIR
    / "X_train.csv",
    index=False
)

X_validation_processed.to_csv(
    FEATURE_OUTPUT_DIR
    / "X_validation.csv",
    index=False
)

X_test_processed.to_csv(
    FEATURE_OUTPUT_DIR
    / "X_test.csv",
    index=False
)

X_prediction_processed.to_csv(
    FEATURE_OUTPUT_DIR
    / "X_prediction.csv",
    index=False
)


# ------------------------------------------------------------
# Save prediction identifiers
# ------------------------------------------------------------

prediction_identifier_columns = [
    column
    for column in [
        "title",
        "release_date",
        "release_year"
    ]
    if column in prediction.columns
]

prediction_identifiers = (
    prediction[
        prediction_identifier_columns
    ]
    .copy()
)

prediction_identifiers.to_csv(
    FEATURE_OUTPUT_DIR
    / "prediction_identifiers.csv",
    index=False
)


# ------------------------------------------------------------
# Save historical split reference
# ------------------------------------------------------------

split_reference_frames = []

for split_name, df in {
    "train": train_df,
    "validation": validation_df,
    "test": test_df
}.items():

    reference_columns = [
        column
        for column in [
            "title",
            "release_date",
            "release_year",
            "worldwide_box_office"
        ]
        if column in df.columns
    ]

    temp = (
        df[reference_columns]
        .copy()
    )

    temp["split"] = split_name

    split_reference_frames.append(
        temp
    )


split_reference = pd.concat(
    split_reference_frames,
    ignore_index=True
)

split_reference.to_csv(
    METADATA_OUTPUT_DIR
    / "historical_split_reference.csv",
    index=False
)


print("=" * 80)
print("MODEL-READY FEATURES SAVED")
print("=" * 80)

print("\nFeature output directory:")
print(FEATURE_OUTPUT_DIR)

print("\nFiles saved:")
print(" - X_train.csv")
print(" - X_validation.csv")
print(" - X_test.csv")
print(" - X_prediction.csv")
print(" - prediction_identifiers.csv")
print(" - historical_split_reference.csv")

MODEL-READY FEATURES SAVED

Feature output directory:
C:\Users\bongo\OneDrive\Desktop\GitHub\Movie Analysis Predictor\Data\Model_Ready\Features

Files saved:
 - X_train.csv
 - X_validation.csv
 - X_test.csv
 - X_prediction.csv
 - prediction_identifiers.csv
 - historical_split_reference.csv


## 19. SAVE TARGETS, PREPROCESSOR AND METADATA

In [21]:
import json
import joblib


# ------------------------------------------------------------
# Save regression targets
# ------------------------------------------------------------

pd.DataFrame({
    "log_worldwide_box_office": y_train_reg
}).reset_index(drop=True).to_csv(
    TARGET_OUTPUT_DIR
    / "y_train_regression.csv",
    index=False
)

pd.DataFrame({
    "log_worldwide_box_office": y_validation_reg
}).reset_index(drop=True).to_csv(
    TARGET_OUTPUT_DIR
    / "y_validation_regression.csv",
    index=False
)

pd.DataFrame({
    "log_worldwide_box_office": y_test_reg
}).reset_index(drop=True).to_csv(
    TARGET_OUTPUT_DIR
    / "y_test_regression.csv",
    index=False
)


# ------------------------------------------------------------
# Save classification targets
# ------------------------------------------------------------

pd.DataFrame({
    "blockbuster": y_train_cls
}).reset_index(drop=True).to_csv(
    TARGET_OUTPUT_DIR
    / "y_train_classification.csv",
    index=False
)

pd.DataFrame({
    "blockbuster": y_validation_cls
}).reset_index(drop=True).to_csv(
    TARGET_OUTPUT_DIR
    / "y_validation_classification.csv",
    index=False
)

pd.DataFrame({
    "blockbuster": y_test_cls
}).reset_index(drop=True).to_csv(
    TARGET_OUTPUT_DIR
    / "y_test_classification.csv",
    index=False
)


# ------------------------------------------------------------
# Save fitted preprocessing pipeline
# ------------------------------------------------------------

PREPROCESSOR_PATH = (
    METADATA_OUTPUT_DIR
    / "preprocessor.joblib"
)

joblib.dump(
    preprocessor,
    PREPROCESSOR_PATH
)


# ------------------------------------------------------------
# Save project modelling metadata
# ------------------------------------------------------------

model_ready_metadata = {

    "regression_target":
        REGRESSION_TARGET,

    "classification_target":
        CLASSIFICATION_TARGET,

    "blockbuster_definition":
        "Top 10% of training-period worldwide box office",

    "blockbuster_threshold_usd":
        float(BLOCKBUSTER_THRESHOLD),

    "training_end_year":
        TRAIN_END_YEAR,

    "validation_start_year":
        VALIDATION_START_YEAR,

    "validation_end_year":
        VALIDATION_END_YEAR,

    "test_start_year":
        TEST_START_YEAR,

    "test_end_year":
        TEST_END_YEAR,

    "numeric_features":
        final_numeric_features,

    "categorical_features":
        final_categorical_features,

    "raw_feature_count":
        len(final_model_features),

    "processed_feature_count":
        len(processed_feature_names),

    "processed_features":
        processed_feature_names,

    "redundant_feature_removed":
        "release_quarter",

    "deferred_issue":
        (
            "Historical model dataset currently contains "
            "18 primary genres while the upstream cleaned "
            "source contains 19. Reconcile upstream genre "
            "coverage later, rerun Feature Engineering, "
            "then rerun this notebook."
        )
}


METADATA_PATH = (
    METADATA_OUTPUT_DIR
    / "model_ready_metadata.json"
)

with open(
    METADATA_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        model_ready_metadata,
        file,
        indent=4
    )


print("=" * 80)
print("TARGETS AND METADATA SAVED")
print("=" * 80)

print("\nTarget directory:")
print(TARGET_OUTPUT_DIR)

print("\nPreprocessor:")
print(PREPROCESSOR_PATH)

print("\nMetadata:")
print(METADATA_PATH)

print(
    f"\nTraining-derived blockbuster threshold: "
    f"${BLOCKBUSTER_THRESHOLD:,.0f}"
)

TARGETS AND METADATA SAVED

Target directory:
C:\Users\bongo\OneDrive\Desktop\GitHub\Movie Analysis Predictor\Data\Model_Ready\Targets

Preprocessor:
C:\Users\bongo\OneDrive\Desktop\GitHub\Movie Analysis Predictor\Data\Model_Ready\Metadata\preprocessor.joblib

Metadata:
C:\Users\bongo\OneDrive\Desktop\GitHub\Movie Analysis Predictor\Data\Model_Ready\Metadata\model_ready_metadata.json

Training-derived blockbuster threshold: $183,907,650


## 20. MODEL-READY DATA PREPARATION SUMMARY

In [22]:
model_ready_summary = pd.DataFrame({

    "item": [
        "Historical observations",
        "Regression target",
        "Classification target",
        "Blockbuster definition",
        "Split strategy",
        "Redundant feature removed",
        "Missing-value treatment",
        "Categorical encoding",
        "Numerical scaling",
        "Structural history indicators",
        "Budget missingness indicator",
        "Processed training features",
        "Prediction feature matrix",
        "Data leakage protection",
        "Deferred correction"
    ],

    "decision": [
        f"{len(historical):,} movies",

        "log_worldwide_box_office",

        "blockbuster",

        (
            "Top 10% of training-period "
            f"box office "
            f"(threshold ${BLOCKBUSTER_THRESHOLD:,.0f})"
        ),

        (
            f"Chronological: train through "
            f"{TRAIN_END_YEAR}, validation "
            f"{VALIDATION_START_YEAR}–"
            f"{VALIDATION_END_YEAR}, test "
            f"{TEST_START_YEAR}–"
            f"{TEST_END_YEAR}"
        ),

        "release_quarter removed; release_month retained",

        "Median numerical and most-frequent categorical imputation fitted on training data only",

        "One-hot encoding with unknown-category handling",

        "StandardScaler fitted on training data only",

        "Director, lead-star and company prior-history indicators retained",

        "budget_missing indicator retained",

        f"{X_train_processed.shape[1]} processed predictors",

        f"{X_prediction_processed.shape[0]} target movies prepared",

        "Preprocessor fitted only on chronological training data",

        (
            "Reconcile 18/19 genre discrepancy later; "
            "rerun Feature Engineering and this notebook afterward"
        )
    ]
})


print("=" * 80)
print("MODEL-READY DATA PREPARATION — COMPLETE")
print("=" * 80)

display(model_ready_summary)


print("\nSaved model-ready structure:")

print("""
Data/
└── Model_Ready/
    ├── Features/
    │   ├── X_train.csv
    │   ├── X_validation.csv
    │   ├── X_test.csv
    │   ├── X_prediction.csv
    │   └── prediction_identifiers.csv
    │
    ├── Targets/
    │   ├── y_train_regression.csv
    │   ├── y_validation_regression.csv
    │   ├── y_test_regression.csv
    │   ├── y_train_classification.csv
    │   ├── y_validation_classification.csv
    │   └── y_test_classification.csv
    │
    └── Metadata/
        ├── historical_split_reference.csv
        ├── preprocessor.joblib
        └── model_ready_metadata.json
""")


print(
    "STATUS: MODEL-READY DATA PREPARATION COMPLETE."
)

print(
    "\nDEFERRED: Resolve the 18/19 genre discrepancy "
    "before final model training/presentation, then "
    "rerun the affected pipeline."
)

MODEL-READY DATA PREPARATION — COMPLETE


,item,decision
0,Historical observations,"7,479 movies"
1,Regression target,log_worldwide_box_office
2,Classification target,blockbuster
3,Blockbuster definition,Top 10% of training-period box office (thresho...
4,Split strategy,"Chronological: train through 2015, validation ..."
5,Redundant feature removed,release_quarter removed; release_month retained
6,Missing-value treatment,Median numerical and most-frequent categorical...
7,Categorical encoding,One-hot encoding with unknown-category handling
8,Numerical scaling,StandardScaler fitted on training data only
9,Structural history indicators,"Director, lead-star and company prior-history ..."



Saved model-ready structure:

Data/
└── Model_Ready/
    ├── Features/
    │   ├── X_train.csv
    │   ├── X_validation.csv
    │   ├── X_test.csv
    │   ├── X_prediction.csv
    │   └── prediction_identifiers.csv
    │
    ├── Targets/
    │   ├── y_train_regression.csv
    │   ├── y_validation_regression.csv
    │   ├── y_test_regression.csv
    │   ├── y_train_classification.csv
    │   ├── y_validation_classification.csv
    │   └── y_test_classification.csv
    │
    └── Metadata/
        ├── historical_split_reference.csv
        ├── preprocessor.joblib
        └── model_ready_metadata.json

STATUS: MODEL-READY DATA PREPARATION COMPLETE.

DEFERRED: Resolve the 18/19 genre discrepancy before final model training/presentation, then rerun the affected pipeline.
